# Codebook 01 — QAOA / MaxCut

PACA DOCENCIA · Quantum AI for Queens · edición 2026-11-13. Material pedagógico nuevo preparado para revisión editorial JJBV. Sin copiar notebooks de terceros. No requiere hardware cuántico.


Para un grafo con aristas E, C(z) es el número de aristas cuyos extremos tienen bits distintos. QAOA p=1 aplica exp(-i gamma C) a |+>^n y después exp(-i beta sum X_i). Comparamos contra enumeración exacta; no afirmamos ventaja cuántica.


In [ ]:
import numpy as np
from itertools import product
edges = ((0,1),(1,2))
n=3
strings=list(product((0,1),repeat=n))
def cut_value(z):
    return sum(int(z[i]!=z[j]) for i,j in edges)
costs=np.array([cut_value(z) for z in strings],dtype=float)
optimum=float(max(costs))
assert optimum==2 and np.isclose(costs.mean(),1)
print('MaxCut exacto:',optimum)


In [ ]:
def qaoa_probabilities(gamma,beta):
    psi=np.exp(-1j*gamma*costs)/np.sqrt(2**n)
    for q in range(n):
        bit=1<<(n-1-q)
        nxt=np.zeros_like(psi)
        for k in range(2**n):
            nxt[k]=np.cos(beta)*psi[k]-1j*np.sin(beta)*psi[k^bit]
        psi=nxt
    return np.abs(psi)**2
def expected_cut(gamma,beta):
    return float(np.dot(qaoa_probabilities(gamma,beta),costs))
values=[(g,b,expected_cut(g,b)) for g in np.linspace(0,np.pi,13)
        for b in np.linspace(0,np.pi/2,13)]
best=max(values,key=lambda row:row[2])
assert np.isclose(qaoa_probabilities(*best[:2]).sum(),1)
assert 0<=best[2]<=optimum+1e-12
assert np.isclose(expected_cut(0,0),1)
print('Mejor esperanza en rejilla:',round(best[2],6))


Ejercicio: cambiar el grafo a un triángulo y registrar código, resultados, traza e interpretación. La búsqueda en rejilla no prueba optimalidad global de QAOA.
